# ICD 341 LABS 9-13 
**NAME: MAINZA MUYA**    
**STUDENT NUMBER : 202406802**


## Lab 1: Bootstrap & resampling 


### Questions 
#### (1) Bootstrap the mean of [3, 4, 5, 7, 8, 9] and report a 95% percentile interval,To bootstrap the mean: 

In [1]:
import numpy as np

data = np.array([3, 4, 5, 7, 8, 9])
np.random.seed(42)
bootstrap_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(10000)]
ci_lower, ci_upper = np.percentile(bootstrap_means, [2.5, 97.5])

print(f"95% Percentile Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")

95% Percentile Interval: [4.3333, 7.6667]


#### (2) Bootstrap the median of the same data

In [2]:
bootstrap_medians = [np.median(np.random.choice(data, size=len(data), replace=True)) for _ in range(10000)]
median_ci_lower, median_ci_upper = np.percentile(bootstrap_medians, [2.5, 97.5])

print(f"Median 95% Percentile Interval: [{median_ci_lower:.4f}, {median_ci_upper:.4f}]")

Median 95% Percentile Interval: [3.5000, 8.5000]


#### (3) Simulate a parametric bootstrap from a fitted normal model

In [3]:
# Estimate parameters from the original data
mu, std = np.mean(data), np.std(data, ddof=1)

# Simulate parametric bootstrap samples from N(mu, std)
param_bootstrap_samples = np.random.normal(mu, std, size=(10000, len(data)))
param_bootstrap_means = np.mean(param_bootstrap_samples, axis=1)
param_ci = np.percentile(param_bootstrap_means, [2.5, 97.5])

print(f"Parametric Bootstrap 95% CI for Mean: [{param_ci[0]:.4f}, {param_ci[1]:.4f}]")

Parametric Bootstrap 95% CI for Mean: [4.0970, 7.9003]


#### (4) Explain one way data leakage can occur during model validation.        
Data leakage occurs when information from outside the training dataset is improperly used to create or validate a machine learning model. One common way this happens is during data preprocessing (such as normalization, standard scaling, or imputation) before cross-validation.

## Lab 2: Bayesian MCMC 

 ### Questions 

#### (1) Implement random-walk MH for a standard normal target.

In [4]:
import numpy as np

def log_target(x):
    # Unnormalized log pdf of standard normal: -0.5 * x^2
    return -0.5 * x**2

def metropolis_hastings(num_samples=10000, proposal_std=0.5):
    samples = np.empty(num_samples)
    current_x = 0.0
    current_log_prob = log_target(current_x)
    
    for i in range(num_samples):
        # Propose a new state using a normal random walk
        proposed_x = np.random.normal(current_x, proposal_std)
        proposed_log_prob = log_target(proposed_x)
        
        # Calculate acceptance probability in log space
        log_acceptance_ratio = proposed_log_prob - current_log_prob
        
        if np.log(np.random.uniform(0, 1)) < log_acceptance_ratio:
            current_x = proposed_x
            current_log_prob = proposed_log_prob
            
        samples[i] = current_x
        
    return samples

mh_samples = metropolis_hastings()

#### (2) Estimate a Bernoulli probability with a Beta(2,2) prior after observing 8 successes in 12 trials.

Due to beta-binomial conjugacy, a Beta prior combined with binomial observations yields an updated Beta posterior:Prior: $\text{Beta}(\alpha_0, \beta_0) = \text{Beta}(2, 2)$Likelihood / Data: $s = 8$ successes, $f = 12 - 8 = 4$ failures.Posterior: $\text{Beta}(\alpha_0 + s, \beta_0 + f) = \text{Beta}(2 + 8, 2 + 4) = \text{Beta}(10, 6)$.The posterior expected value (mean) for the probability of success is:$$\hat{p} = \frac{\alpha}{\alpha + \beta} = \frac{10}{10 + 6} = \frac{10}{16} = 0.625$$

#### (3) Explain trace plots, ACF, R-hat, and ESS.
  * Trace Plots: Line plots of the MCMC parameter values across sampling iterations. They help visually diagnose mixing (whether the chain moves freely around the parameter space) and stationarity (whether the chain has converged to the target distribution, often resembling a stationary "fuzzy caterpillar").
  * ACF (Autocorrelation Function): Measures the linear correlation between samples of a chain at different lag steps. High autocorrelation indicates that successive samples are sluggishly dependent, meaning fewer effective independent samples are gathered per iteration.
  * R-hat ($\hat{R}$ / Gelman-Rubin Diagnostic): A convergence diagnostic that compares the variance between multiple chains to the variance within the chains. Values close to 1.0 (typically $< 1.05$) suggest that the chains have successfully mixed and converged to the same target distribution.
  * ESS (Effective Sample Size): Estimates the number of independent samples equivalent to the correlated samples generated by the MCMC algorithm. A higher ESS indicates lower parameter estimation error.

#### (4) Write a PyMC model for normally distributed observations

In [1]:
import pymc as pm
import numpy as np

# Generate synthetic data
np.random.seed(0)
true_mu, true_sigma = 5.0, 2.0
data = np.random.normal(true_mu, true_sigma, 100)

with pm.Model() as normal_model:
    # Priors for unknown parameters
    mu = pm.Normal("mu", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)
    
    # Likelihood (sampling distribution) of observations
    likelihood = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=data)
    
    # Draw posterior samples
    trace = pm.sample(draws=1000, tune=1000, return_inferencedata=True)

g++ not available, if using conda: `conda install gxx`
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu, sigma]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 10 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


## Lab 3: Advanced simulation

### Questions 

#### (1) Generate 5,000 exponential samples using inverse transform with rate 3.
The cumulative distribution function (CDF) of an exponential distribution is $F(x) = 1 - e^{-\lambda x}$. Setting $U = 1 - e^{-\lambda x}$ where $U \sim \text{Uniform}(0, 1)$ and solving for $x$ yields the inverse transform formula:$$x = -\frac{\ln(1 - U)}{\lambda}$$Since $1 - U$ is identically distributed as $U$, this simplifies to:$$x = -\frac{\ln(U)}{\lambda}$$

In [8]:
import numpy as np

np.random.seed(42)
rate = 3.0
u = np.random.uniform(0, 1, 5000)
exponential_samples = -np.log(u) / rate

#### (2) Sample from a categorical distribution with probabilities [0.1, 0.6, 0.3].

In [7]:
probabilities = [0.1, 0.6, 0.3]
categories = [0, 1, 2]

# Sample 10 items based on the specified categorical probabilities
categorical_samples = np.random.choice(categories, size=10, p=probabilities)

#### (3) Implement rejection sampling for Beta(2,2).
A $\text{Beta}(2,2)$ density is bounded on $[0,1]$ with a maximum density value at $x = 0.5$, where $f(0.5) = \frac{\Gamma(4)}{\Gamma(2)\Gamma(2)} (0.5)^1 (0.5)^1 = 6 \times 0.25 = 1.5$. We can use a $\text{Uniform}(0,1)$ proposal distribution $g(x) = 1$, with an envelope constant $c = 1.5$.

In [6]:
def beta_22_pdf(x):
    return 6 * x * (1 - x)

def sample_beta_22(num_samples=1000):
    samples = []
    c = 1.5 # Maximum value of Beta(2,2) PDF
    
    while len(samples) < num_samples:
        u1 = np.random.uniform(0, 1)
        u2 = np.random.uniform(0, 1)
        
        # Acceptance condition: u2 <= f(u1) / (c * g(u1))
        if u2 <= beta_22_pdf(u1) / c:
            samples.append(u1)
            
    return np.array(samples)

accepted_samples = sample_beta_22(1000)

#### (4) Describe the four main particle-filter steps.
1. Prediction (Initialization / Proposal): Propagate existing particles forward in time using the system's dynamic state transition model, accounting for process noise.
2. Update (Weighting): Evaluate the incoming sensor/observation data against each predicted particle, assigning an importance weight based on the observation likelihood function ($w_i \propto p(y_t \mid x_t^{(i)})$).
3. Normalization: Normalize the assigned particle weights so that they sum to 1.0, creating a discrete probability distribution over the particles.
4. Resampling: Eliminate particles with negligible weights and duplicate particles with high weights to mitigate the problem of weight degeneracy (where all but one particle lose significance over time).

(5) Explain what a non-uniform SBC rank histogram may indicate.
Simulation-Based Calibration (SBC) checks whether a Bayesian model and its inference algorithm are implemented correctly by checking if posterior ranks are uniformly distributed. A non-uniform rank histogram indicates algorithmic bias, implementation bugs, or software miscalibration:

* U-shaped histogram: Indicates underdispersion (posterior intervals are too narrow, meaning the model is overly confident).

* Dome or inverted U-shape: Indicates overdispersion (posterior intervals are too wide, meaning the model is under-confident).

* Asymmetric skew (leaning left or right): Indicates structural bias or parameter estimation drift (e.g., the sampler systematically overestimates or underestimates a parameter).

## Lab 4: Testing, estimators, packaging

### Questions 

#### (1) Write a function for the median and tests for a known result and empty input.

In [22]:
import numpy as np
import pytest

def compute_median(arr):
    if len(arr) == 0:
        raise ValueError("Input array cannot be empty.")
    return float(np.median(arr))

# Tests
def test_compute_median_known():
    assert compute_median([1, 3, 2, 5, 4]) == 3.0

def test_compute_median_empty():
    with pytest.raises(ValueError):
        compute_median([])

ModuleNotFoundError: No module named 'pytest'

#### (2) Write tests that a probability estimate lies in [0,1].

In [21]:
def test_probability_bounds():
    # Mock estimator function returning a list of probability predictions
    def get_probabilities():
        return [0.0, 0.45, 0.99, 1.0]
    
    probs = get_probabilities()
    for p in probs:
        assert 0.0 <= p <= 1.0, f"Probability {p} is out of bounds [0, 1]"

#### (3) Implement a transformer that scales each column by its standard deviation.

In [20]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class ColumnStdScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = np.asarray(X)
        # Calculate standard deviation per column, avoiding division by zero
        self.scales_ = np.std(X, axis=0)
        self.scales_[self.scales_ == 0.0] = 1.0
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return X / self.scales_

#### (4) Create a minimal pyproject.toml and explain editable installation

In [12]:
Ini, TOML
[build-system]
requires = ["setuptools>=61.0.0"]
build-backend = "setuptools.build_meta"

[project]
name = "my_lab_package"
version = "0.1.0"
description = "A minimal package for ICD 341 labs"
readme = "README.md"
requires-python = ">=3.8"
dependencies = [
    "numpy",
    "scikit-learn"
]

SyntaxError: cannot assign to expression here. Maybe you meant '==' instead of '='? (939721193.py, line 4)

## Lab 5: Performance and scalability

### Questions 

#### (1) Vectorize the calculation $x^{2}+2x+1$ for one million values.

In [ ]:
import numpy as np

# Generate one million values
x = np.arange(1_000_000, dtype=float)

# Vectorized operation using NumPy C-optimized arrays
result = x**2 + 2 * x + 1 

#### (2) Write a Numba function that sums the squares.

In [ ]:
from numba import jit
import numpy as np

@jit(nopython=True)
def sum_of_squares(arr):
    total = 0.0
    for i in range(arr.shape[0]):
        total += arr[i] ** 2
    return total

# Example usage
data_arr = np.linspace(1, 100, 1000)
result = sum_of_squares(data_arr)

#### (3) Bootstrap a sample mean in parallel using joblib

In [ ]:
from joblib import Parallel, delayed
import numpy as np

data = np.array([3, 4, 5, 7, 8, 9])
n_bootstraps = 10000

def single_bootstrap(arr):
    resampled = np.random.choice(arr, size=len(arr), replace=True)
    return np.mean(resampled)

# Run bootstrap iterations in parallel across multiple CPU cores
bootstrap_means = Parallel(n_jobs=-1)(
    delayed(single_bootstrap)(data) for _ in range(n_bootstraps)
)

#### (4) Use Dask Array to calculate the mean of a large chunked array.

In [ ]:
import dask.array as da

# Create a large chunked Dask array (e.g., size 10,000,000 chunked into blocks of 1,000,000)
x = da.arange(10_000_000, chunks=(1_000_000,))

# Calculate the mean lazily and compute the result
mean_value = x.mean().compute()
print(f"Dask Array Mean: {mean_value}")

Dask Array Mean: 4999999.5


#### (5) Explain why benchmark timing should exclude Numba's first compilation call.
When you call a Numba @jit(nopython=True) decorated function for the very first time, Numba triggers Just-In-Time (JIT) compilation, which analyzes types, generates machine code via LLVM, and links the compiled function. This initial compilation step incurs a significant one-time overhead cost.

If you include this first call in your benchmark timings, you are measuring Numba's compiler latency rather than the optimized execution speed of the compiled machine code. Subsequent calls bypass compilation and execute at native C speeds, which is what benchmarks are designed to measure.